# PMAPS Workshop: IDAES-GTEP, Session 2

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/agmoore4/idaes-gtep.git/pmaps-123?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsource%2Ftutorials%2F123bus%2Ftutorial_123bus.ipynb)

Welcome! In this tutorial, we will demonstrate using IDAES Generation and Transmission Expansion Planning (GTEP) with a more complex case--a 123-bus system in Texas.

As we step through the notebook, you should notice that all the steps to set up and solve a model are the same as in the much simpler 5-bus case. The only major difference is the scale of the system reflected in the data files.

In [1]:
# suppressing some logs/warnings
import logging
import warnings
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

## Setting up and solving the model

First, we create our `ExpansionPlanningData` instance and use `load_prescient` to load the relevant data files.

In [2]:
from pathlib import Path
from gtep.gtep_data import ExpansionPlanningData

data_path = Path("../../../../gtep/data/123_Bus_Resil_Week")
# data_path = Path("../../../../gtep/data/5bus")

rep_days = [
    "2019-01-28 00:00",
    # "2019-07-05 00:00",
]
rep_weights = {
    "2019-01-28 00:00": 365,
    # "2019-07-05 00:00": 155,
}

data_object = ExpansionPlanningData(
    stages=1,
    num_reps=1,
    num_commit=1,
    num_dispatch=2,
    duration_representative_period=0.5,  # keeping dispatch periods 15 min
)
data_object.load_prescient(
    data_path,
    representative_dates=rep_days,
    representative_weights=rep_weights,
)

Interactive Python mode detected; using default matplotlib backend for plotting.
Setting default t0 state in RTS-GMLC parser
INFO: representative_dates and representative_weights are aligned.Continue building the data modeling object...


Next we read in cost data using `DataProcessing`:

In [3]:
from gtep.gtep_data_processing import DataProcessing

bus_data_path = Path(
    "../../../../gtep/data/costs/Bus_data_gen_weights_mappings.csv"
)
cost_data_path = Path(
    "../../../../gtep/data/costs/2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx"
)
ng_cost_path = Path(
    "../../../../gtep/data/costs/Total_Energy_Supply_Disposition_and_Price_Summary.csv"
)

candidate_gens = [
    "Natural Gas_FE",
    "Solar - Utility PV",
    "Land-Based Wind",
]

cost_data = DataProcessing()
cost_data.load_gen_data(
    bus_data_path=bus_data_path,
    cost_data_path=cost_data_path,
    ng_cost_path=ng_cost_path,
    candidate_gens=candidate_gens,
)

Finally, we create and solve the model.

In [4]:
from gtep.gtep_model import ExpansionPlanningModel
from pyomo.environ import SolverFactory, TransformationFactory

mod_object = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False},
)
mod_object.create_model()
mod_object.timer.toc("Finished model build")

TransformationFactory("gdp.bigm").apply_to(mod_object.model)
mod_object.timer.toc("Finished model transformation")

# appsi_highs allows us to set scaling options,
# which often reduces solve times significantly
opt = SolverFactory("appsi_highs")
opt.highs_options["user_objective_scale"] = -4
opt.highs_options["user_bound_scale"] = -8

result = opt.solve(mod_object.model, tee=True)
mod_object.timer.toc("Finished solving");

[    0.00] Creating GTEP Model
[+   0.52] Finished model build
[+   0.63] Finished model transformation
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 12319 rows; 9897 cols; 33754 nonzeros; 3960 integer variables (3481 binary)
Assessing costs and bounds after applying user_objective_scale option value of -4 and user_bound_scale option value of -8
Coefficient ranges:
  Matrix  [4e-03, 3e+06]
  Cost    [6e-02, 9e+05]
  Bound   [4e-03, 2e+01]
  RHS     [1e-03, 7e+04]
Presolving model
6317 rows, 4594 cols, 17720 nonzeros 0s
5404 rows, 3658 cols, 14759 nonzeros 0s
5401 rows, 3597 cols, 14703 nonzeros 0s
Presolve reductions: rows 5401(-6918); columns 3597(-6300); nonzeros 14703(-19051) 

Solving MIP model with:
   5401 rows
   3597 cols (1735 binary, 0 integer, 0 implied int., 1862 continuous, 0 domain fixed)
   14703 nonzeros
   Thread count 4 (of 8 threads). U

Now we use the `ExpansionPlanningSolution` class to write out the results and generate plots:

In [5]:
from copy import copy
from IPython.display import display, HTML
from gtep.gtep_solution import ExpansionPlanningSolution

def display_plotly_as_html(fig, title=None):
    if title is not None:
        fig = copy(fig)
        fig.update_layout(title=title)
    fig_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    return display(HTML(fig_html))

# create solution object and write out to json
soln = ExpansionPlanningSolution(data_path)
soln_path = Path("soln")
soln.save_results_in_json_files(mod_object, soln_path)

# perform plotting
pie = soln.create_plots("combined", soln_path, data_path, "piechart", savefig=False)
stackgraph = soln.create_stackgraph(soln_path, rep_days, savefig=False)

display_plotly_as_html(pie)
display_plotly_as_html(stackgraph)

## Experiments

### Experiment 1

Experiment 1 contents

### Experiment 2

Experiment 2 contents